In [13]:
import base64
import csv
import itertools
import os
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    import yaml  # pip install pyyaml
except ImportError:
    yaml = None

try:
    from tqdm import tqdm  # pip install tqdm
except ImportError:
    tqdm = None


# ============================================================
# CONFIG (your paths + switches)
# ============================================================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
REPO_LIST_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\URL_List_Instru.csv"
)

OUT_WORKFLOW_MAP_CSV = REPO_LIST_CSV.parent / "workflow_instru_map.csv"
OUT_RUN_METRICS_CSV = REPO_LIST_CSV.parent / "run_instru_metrics.csv"

# Which workflows to pull runs for:
#   - "INSTRU_ONLY": only workflows that look like they contain instrumentation
#   - "ALL": fetch runs for all workflows found
RUN_WORKFLOW_MODE = "INSTRU_ONLY"  # or "ALL"

# Heavy option: fetch jobs for every run (needed for instrumentation-only conclusion + durations).
FETCH_JOBS_FOR_EACH_RUN = True

# NEW: focus only on default branch runs (server-side filter + safety check)
DEFAULT_BRANCH_ONLY = True

# If you want to cap history (optional). None means "full possible history".
MAX_RUNS_PER_WORKFLOW: Optional[int] = None

# Optional date filter (ISO8601 like "2024-01-01T00:00:00Z"). None means no filter.
RUN_CREATED_AT_AFTER: Optional[str] = None

# ----------- API robustness knobs (important) -----------
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60

MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60

# Safety: if GitHub returns a weird repeated page, break pagination after too many pages.
MAX_PAGES_PER_LIST = 2000


# ============================================================
# Classification patterns (tune as needed)
# ============================================================

COMMUNITY_ACTION_RE = re.compile(
    r"(reactivecircus/android-emulator-runner|android-emulator-runner|usuiat/android-emulator-runner|"
    r"circus/android-emulator-runner)",
    re.IGNORECASE,
)

CUSTOM_EMU_RE = re.compile(
    r"(\bavdmanager\b|\bsdkmanager\b.*system-images|\bemulator\b.*\s-avd\b|\bnohup\s+emulator\b|"
    r"\badb\s+wait-for-device\b|\blaunch(es|ing)?\s+emulator\b)",
    re.IGNORECASE,
)

GMD_RE = re.compile(r"(\bmanagedDeviceCheck\b|\ballDevicesCheck\b|\bmanagedDevice\b)", re.IGNORECASE)

THIRD_PARTY_RE = re.compile(
    r"(\bfirebase\s+test\s+android\s+run\b|\bgcloud\s+firebase\s+test\b|"
    r"\baws\b.*\bdevice\s*farm\b|\bdevicefarm\b|"
    r"\bbrowserstack\b|\bsaucelabs\b|\bkobiton\b|\bperfecto\b|\bbitbar\b|"
    r"\bgenymotion\b|\bapp(etize|etize)\b)",
    re.IGNORECASE,
)

GRADLE_CMD_RE = re.compile(r"(^|\s)(\./gradlew|\bgradle\b)\s+([^\n\r#;]+)", re.IGNORECASE)
ADB_INSTR_RE = re.compile(r"\badb\s+shell\s+am\s+instrument\b", re.IGNORECASE)
FIREBASE_TEST_RE = re.compile(r"\b(firebase\s+test\s+android\s+run|gcloud\s+firebase\s+test)\b", re.IGNORECASE)

ANDROID_TEST_TASK_RE = re.compile(
    r"\b(connected\w*AndroidTest|connectedCheck|deviceCheck|\w*AndroidTest|managedDeviceCheck|allDevicesCheck)\b",
    re.IGNORECASE,
)

INSTRU_STEP_NAME_RE = re.compile(
    r"(instrument|connected.*androidtest|androidtest|manageddevice|gmd|emulator runner|"
    r"firebase test|test lab|device farm|uiautomator|espresso)",
    re.IGNORECASE,
)

INSTRU_JOB_NAME_RE = re.compile(
    r"(instrument|androidtest|connected|manageddevice|gmd|emulator|firebase|test lab|device farm)",
    re.IGNORECASE,
)


# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None


def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None


def safe_yaml_load(text: str) -> Optional[Dict]:
    if yaml is None:
        return None
    try:
        return yaml.safe_load(text)
    except Exception:
        return None


def b64_to_text(content_b64: str) -> Optional[str]:
    try:
        return base64.b64decode(content_b64).decode("utf-8", errors="ignore")
    except Exception:
        return None


def load_tokens_from_env_file(env_path: Path) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)

    if not tokens:
        raise ValueError(
            f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=... through GITHUB_TOKEN_6=..."
        )
    return tokens


def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = (repo_url_or_fullname or "").strip()
    if not s:
        return None
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s

    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full
    return None


def read_repo_list(csv_path: Path) -> List[Tuple[str, str]]:
    """
    Returns list of (repo_url_or_value, full_name)
    Handles either:
      - first column is URL/full_name
      - a column named url/repo/full_name exists
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Repo list CSV not found: {csv_path}")

    out: List[Tuple[str, str]] = []
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return out

        header = [c.strip().lower() for c in rows[0]]
        col_idx = 0
        has_header = any(("url" in c or "repo" in c or "full" in c) for c in header)

        if has_header:
            for i, name in enumerate(header):
                if name in ("url", "repo_url", "repo", "repository", "full_name"):
                    col_idx = i
                    break

        start = 1 if has_header else 0

        for r in rows[start:]:
            if not r:
                continue
            if col_idx >= len(r):
                continue
            val = (r[col_idx] or "").strip()
            if not val:
                continue
            full = parse_repo_full_name(val)
            if not full:
                continue
            out.append((val, full))
    return out


# ============================================================
# CSV resume helpers
# ============================================================
def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys


def _read_csv_header(csv_path: Path) -> Optional[List[str]]:
    try:
        with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            r = csv.reader(f)
            return next(r, None)
    except Exception:
        return None


def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> Path:
    """
    If file doesn't exist: create it with header.
    If it exists but header mismatches: write to a new versioned file and return that path.
    """
    if not csv_path.exists():
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        with csv_path.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
        return csv_path

    existing = _read_csv_header(csv_path) or []
    if [c.strip() for c in existing] == fieldnames:
        return csv_path

    # Header mismatch -> create a new file to avoid corrupting an existing CSV
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    new_path = csv_path.with_name(f"{csv_path.stem}_v2_{stamp}{csv_path.suffix}")
    new_path.parent.mkdir(parents=True, exist_ok=True)
    with new_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
    print(f"[warn] CSV header mismatch for {csv_path.name}. Writing to new file: {new_path.name}")
    return new_path


def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)


# ============================================================
# GitHub API client with token rotation + bounded retries
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "instru-miner-v3/1.1",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(
        self,
        method: str,
        url: str,
        params: Optional[Dict] = None
    ) -> Optional[Union[Dict, List]]:
        last_status = None

        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(
                    method,
                    url,
                    params=params,
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                )
            except requests.exceptions.RequestException as e:
                print(f"[net] {method} {url} attempt {attempt}/{MAX_RETRIES_PER_REQUEST}: {e}")
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                print(f"[5xx] {method} {url} -> {resp.status_code} attempt {attempt}/{MAX_RETRIES_PER_REQUEST}")
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                print(f"[error] {method} {url} -> {resp.status_code}")
                print((resp.text or "")[:500])
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})

            data = self.request_json("GET", url, params=p)
            if not data:
                return

            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return

            for it in items:
                yield it

            if isinstance(items, list) and len(items) < 100:
                return

            page += 1

        print(f"[warn] pagination safety stop: exceeded {MAX_PAGES_PER_LIST} pages for {url}")


# ============================================================
# Repo metadata (default branch)
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()


# ============================================================
# Workflow YAML fetch + classification
# ============================================================
def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    return list(gh.paginate(url, params={}, item_key="workflows"))


def fetch_workflow_yaml_text(gh: GitHubClient, full_name: str, workflow_path: str) -> Optional[str]:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return None

    if data.get("encoding") == "base64" and data.get("content"):
        return b64_to_text(data["content"])

    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text
        except requests.exceptions.RequestException:
            return None
    return None


def extract_run_blocks(yobj: Dict) -> List[str]:
    runs: List[str] = []
    if not isinstance(yobj, dict):
        return runs
    jobs = yobj.get("jobs")
    if not isinstance(jobs, dict):
        return runs
    for _, job in jobs.items():
        if not isinstance(job, dict):
            continue
        steps = job.get("steps")
        if not isinstance(steps, list):
            continue
        for st in steps:
            if not isinstance(st, dict):
                continue
            r = st.get("run")
            if isinstance(r, str) and r.strip():
                runs.append(r)
    return runs


def extract_uses(yobj: Dict) -> List[str]:
    uses: List[str] = []
    if not isinstance(yobj, dict):
        return uses
    jobs = yobj.get("jobs")
    if not isinstance(jobs, dict):
        return uses
    for _, job in jobs.items():
        if not isinstance(job, dict):
            continue
        steps = job.get("steps")
        if not isinstance(steps, list):
            continue
        for st in steps:
            if not isinstance(st, dict):
                continue
            u = st.get("uses")
            if isinstance(u, str) and u.strip():
                uses.append(u)
    return uses


def classify_workflow(yaml_text: str) -> Tuple[Set[str], Set[str], Set[str], bool]:
    styles: Set[str] = set()
    inv_types: Set[str] = set()
    hints: Set[str] = set()

    if COMMUNITY_ACTION_RE.search(yaml_text):
        styles.add("Emu_Community_Action")
    if CUSTOM_EMU_RE.search(yaml_text):
        styles.add("Emu_Custom")
    if GMD_RE.search(yaml_text):
        styles.add("GMD")
    if THIRD_PARTY_RE.search(yaml_text):
        styles.add("ThirdParty")

    yobj = safe_yaml_load(yaml_text)
    if yobj:
        for u in extract_uses(yobj):
            if COMMUNITY_ACTION_RE.search(u):
                hints.add(f"uses: {u}"[:240])

        for block in extract_run_blocks(yobj):
            for line in block.splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue

                m = GRADLE_CMD_RE.search(line)
                if m:
                    cmd = (m.group(2) + " " + m.group(3)).strip()
                    if ANDROID_TEST_TASK_RE.search(cmd) or "androidtest" in cmd.lower() or "connected" in cmd.lower():
                        hints.add(cmd[:240])
                        if re.search(r"\bmanagedDeviceCheck\b|\ballDevicesCheck\b", cmd, re.IGNORECASE):
                            inv_types.add("Gradle-GMD")
                        if re.search(r"\bconnected\w*AndroidTest\b|\bconnectedCheck\b|\bdeviceCheck\b", cmd, re.IGNORECASE):
                            inv_types.add("Gradle-ConnectedAndroidTest")

                if ADB_INSTR_RE.search(line):
                    hints.add(line[:240])
                    inv_types.add("ADB-am-instrument")
                if FIREBASE_TEST_RE.search(line):
                    hints.add(line[:240])
                    inv_types.add("Firebase-TestLab")

    for m in GRADLE_CMD_RE.finditer(yaml_text):
        cmd = (m.group(2) + " " + m.group(3)).strip()
        if ANDROID_TEST_TASK_RE.search(cmd):
            hints.add(cmd[:240])
            if re.search(r"\bmanagedDeviceCheck\b|\ballDevicesCheck\b", cmd, re.IGNORECASE):
                inv_types.add("Gradle-GMD")
            if re.search(r"\bconnected\w*AndroidTest\b|\bconnectedCheck\b|\bdeviceCheck\b", cmd, re.IGNORECASE):
                inv_types.add("Gradle-ConnectedAndroidTest")

    if ADB_INSTR_RE.search(yaml_text):
        inv_types.add("ADB-am-instrument")
    if FIREBASE_TEST_RE.search(yaml_text):
        inv_types.add("Firebase-TestLab")

    looks_like_instru = bool(
        styles
        or inv_types
        or ANDROID_TEST_TASK_RE.search(yaml_text)
        or re.search(r"\binstrument(ation)?\b", yaml_text, re.IGNORECASE)
    )

    if not inv_types and styles:
        if "GMD" in styles:
            inv_types.add("Gradle-GMD")
        if "ThirdParty" in styles:
            inv_types.add("ThirdParty-Cloud")
        if "Emu_Community_Action" in styles or "Emu_Custom" in styles:
            inv_types.add("Emulator-Based")

    if not inv_types:
        inv_types.add("UNKNOWN")

    return styles, inv_types, hints, looks_like_instru


# ============================================================
# Runs + Jobs analysis (instrumentation-specific)
# ============================================================
def list_workflow_runs(
    gh: GitHubClient,
    full_name: str,
    workflow_id: int,
    branch: Optional[str] = None,
) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_id}/runs"
    params = {}
    if branch:
        params["branch"] = branch  # server-side filter for default-branch-only
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))


def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


def infer_instru_from_jobs(jobs: List[Dict]) -> Tuple[str, str, str, str, Optional[int], Optional[int], str]:
    if not jobs:
        return "unknown", "none", "", "", None, None, ""

    starts: List[datetime] = []
    ends: List[datetime] = []
    all_labels: Set[str] = set()

    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt:
            starts.append(sdt)
        if edt:
            ends.append(edt)
        labels = j.get("labels")
        if isinstance(labels, list):
            for lab in labels:
                if isinstance(lab, str) and lab.strip():
                    all_labels.add(lab.strip())

    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)

    best_job = None
    best_step = None
    for j in jobs:
        job_name = j.get("name") or ""
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []
        for st in steps:
            step_name = (st.get("name") or "")
            if INSTRU_STEP_NAME_RE.search(step_name) or INSTRU_JOB_NAME_RE.search(job_name):
                concl = st.get("conclusion") or st.get("status") or ""
                if concl:
                    best_job = j
                    best_step = st
                    if concl == "failure":
                        break
        if best_step and (best_step.get("conclusion") == "failure"):
            break

    if best_job and best_step:
        concl = best_step.get("conclusion") or best_step.get("status") or "unknown"
        failed_job = best_job.get("name") or ""
        failed_step = best_step.get("name") or ""
        step_s = iso_to_dt(best_step.get("started_at"))
        step_e = iso_to_dt(best_step.get("completed_at"))
        instru_dur = dt_to_seconds(step_s, step_e)
        if instru_dur is None:
            instru_dur = dt_to_seconds(iso_to_dt(best_job.get("started_at")), iso_to_dt(best_job.get("completed_at")))
        return (
            concl or "unknown",
            "step",
            failed_job if concl == "failure" else "",
            failed_step if concl == "failure" else "",
            instru_dur,
            run_dur,
            ",".join(sorted(all_labels)),
        )

    instru_jobs = []
    for j in jobs:
        job_name = j.get("name") or ""
        if INSTRU_JOB_NAME_RE.search(job_name):
            instru_jobs.append(j)

    if instru_jobs:
        for j in instru_jobs:
            if j.get("conclusion") == "failure":
                instru_dur = dt_to_seconds(iso_to_dt(j.get("started_at")), iso_to_dt(j.get("completed_at")))
                return "failure", "job", j.get("name") or "", "", instru_dur, run_dur, ",".join(sorted(all_labels))
        if any(j.get("conclusion") == "success" for j in instru_jobs):
            j0 = next((j for j in instru_jobs if j.get("conclusion") == "success"), instru_jobs[0])
            instru_dur = dt_to_seconds(iso_to_dt(j0.get("started_at")), iso_to_dt(j0.get("completed_at")))
            return "success", "job", "", "", instru_dur, run_dur, ",".join(sorted(all_labels))

    return "unknown", "none", "", "", None, run_dur, ",".join(sorted(all_labels))


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens)

    repos = read_repo_list(REPO_LIST_CSV)
    if not repos:
        raise RuntimeError(f"No repos read from {REPO_LIST_CSV}")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    workflow_fields = [
        "repo_url",
        "full_name",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "looks_like_instru",
        "styles",
        "invocation_types",
        "invocation_hints",
        "scanned_at_utc",
    ]

    # NEW: default_branch + head_sha included
    run_fields = [
        "full_name",
        "default_branch",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "run_id",
        "run_number",
        "head_sha",
        "created_at",
        "run_started_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "instru_conclusion",
        "instru_detect_method",
        "instru_failed_job",
        "instru_failed_step",
        "instru_duration_seconds",
        "run_duration_seconds",
        "runner_labels_union",
        "extracted_at_utc",
    ]

    out_workflow_map = ensure_csv_header(OUT_WORKFLOW_MAP_CSV, workflow_fields)
    out_run_metrics = ensure_csv_header(OUT_RUN_METRICS_CSV, run_fields)

    existing_run_ids = load_existing_keys(out_run_metrics, "run_id")

    # Build workflow map (in-memory) from CSV if exists
    workflow_map: Dict[Tuple[str, str], Dict] = {}  # (full_name, workflow_id_str) -> info
    if out_workflow_map.exists():
        with out_workflow_map.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            rdr = csv.DictReader(f)
            for row in rdr:
                fn = (row.get("full_name") or "").strip()
                wf_id = (row.get("workflow_id") or "").strip()
                if fn and wf_id:
                    workflow_map[(fn, wf_id)] = row

    repo_iter = repos
    if tqdm is not None:
        repo_iter = tqdm(repos, desc="Repos (workflows)")

    # 1) Scan workflows + classify
    for repo_url, full_name in repo_iter:
        workflows = list_workflows(gh, full_name)
        if not workflows:
            continue

        for wf in workflows:
            wf_id = str(wf.get("id") or "").strip()
            wf_name = (wf.get("name") or "").strip()
            wf_path = (wf.get("path") or "").strip()
            if not wf_id or not wf_path:
                continue

            if (full_name, wf_id) in workflow_map:
                continue

            ytext = fetch_workflow_yaml_text(gh, full_name, wf_path)
            if not ytext:
                row = {
                    "repo_url": repo_url,
                    "full_name": full_name,
                    "workflow_id": wf_id,
                    "workflow_name": wf_name,
                    "workflow_path": wf_path,
                    "looks_like_instru": "unknown",
                    "styles": "",
                    "invocation_types": "UNKNOWN",
                    "invocation_hints": "",
                    "scanned_at_utc": now_utc_iso(),
                }
                append_row(out_workflow_map, workflow_fields, row)
                workflow_map[(full_name, wf_id)] = row
                continue

            styles, inv_types, hints, looks_like = classify_workflow(ytext)

            row = {
                "repo_url": repo_url,
                "full_name": full_name,
                "workflow_id": wf_id,
                "workflow_name": wf_name,
                "workflow_path": wf_path,
                "looks_like_instru": "yes" if looks_like else "no",
                "styles": ";".join(sorted(styles)),
                "invocation_types": ";".join(sorted(inv_types)),
                "invocation_hints": " | ".join(sorted(hints))[:8000],
                "scanned_at_utc": now_utc_iso(),
            }
            append_row(out_workflow_map, workflow_fields, row)
            workflow_map[(full_name, wf_id)] = row

    # 2) Fetch runs (+ jobs) -> instrumentation-specific conclusions + durations
    repo_to_wfs: Dict[str, List[Dict]] = {}
    for (fn, _wf_id), info in workflow_map.items():
        repo_to_wfs.setdefault(fn, []).append(info)

    run_repo_iter = repo_to_wfs.items()
    if tqdm is not None:
        run_repo_iter = tqdm(list(repo_to_wfs.items()), desc="Repos (runs)")

    # Cache default branches so we only hit /repos/{full} once per repo
    default_branch_cache: Dict[str, str] = {}

    for full_name, wfs in run_repo_iter:
        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        selected: List[Dict] = []
        for info in wfs:
            if RUN_WORKFLOW_MODE == "ALL":
                selected.append(info)
            else:
                if (info.get("looks_like_instru") or "").lower() == "yes":
                    selected.append(info)

        for info in selected:
            wf_id = int(info["workflow_id"])
            wf_name = info.get("workflow_name") or ""
            wf_path = info.get("workflow_path") or ""

            runs = list_workflow_runs(
                gh,
                full_name,
                wf_id,
                branch=default_branch if DEFAULT_BRANCH_ONLY else None,
            )

            if MAX_RUNS_PER_WORKFLOW is not None:
                runs = runs[:MAX_RUNS_PER_WORKFLOW]

            for run in runs:
                run_id = str(run.get("id") or "").strip()
                if not run_id:
                    continue
                if run_id in existing_run_ids:
                    continue

                created_at = run.get("created_at") or ""
                if after_dt:
                    cdt = iso_to_dt(created_at)
                    if cdt and cdt < after_dt:
                        continue

                head_branch = run.get("head_branch") or ""
                head_sha = run.get("head_sha") or ""

                # Safety check: enforce default-branch-only even if API filter fails
                if DEFAULT_BRANCH_ONLY and head_branch != default_branch:
                    continue

                run_started_at = run.get("run_started_at") or ""
                status = run.get("status") or ""
                run_conc = run.get("conclusion") or ""
                event = run.get("event") or ""
                html_url = run.get("html_url") or ""
                run_number = run.get("run_number") or ""

                instru_conc = "unknown"
                detect_method = "none"
                failed_job = ""
                failed_step = ""
                instru_dur = None
                run_dur = None
                runner_labels = ""

                if FETCH_JOBS_FOR_EACH_RUN:
                    jobs = list_run_jobs(gh, full_name, int(run_id))
                    instru_conc, detect_method, failed_job, failed_step, instru_dur, run_dur, runner_labels = infer_instru_from_jobs(jobs)

                row = {
                    "full_name": full_name,
                    "default_branch": default_branch,
                    "workflow_id": str(wf_id),
                    "workflow_name": wf_name,
                    "workflow_path": wf_path,
                    "run_id": run_id,
                    "run_number": run_number,
                    "head_sha": head_sha,
                    "created_at": created_at,
                    "run_started_at": run_started_at,
                    "status": status,
                    "run_conclusion": run_conc,
                    "event": event,
                    "head_branch": head_branch,
                    "html_url": html_url,
                    "instru_conclusion": instru_conc,
                    "instru_detect_method": detect_method,
                    "instru_failed_job": failed_job,
                    "instru_failed_step": failed_step,
                    "instru_duration_seconds": "" if instru_dur is None else str(instru_dur),
                    "run_duration_seconds": "" if run_dur is None else str(run_dur),
                    "runner_labels_union": runner_labels,
                    "extracted_at_utc": now_utc_iso(),
                }
                append_row(out_run_metrics, run_fields, row)
                existing_run_ids.add(run_id)

    print("Done.")
    print(f"Workflow map:  {out_workflow_map}")
    print(f"Run metrics:   {out_run_metrics}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n[stopped] Interrupted by user (KeyboardInterrupt).")
        print(f"Partial outputs saved to:")
        print(f"  Workflow map: {OUT_WORKFLOW_MAP_CSV}")
        print(f"  Run metrics:  {OUT_RUN_METRICS_CSV}")


Repos (runs):   3%|▎         | 11/315 [19:48<9:07:22, 108.03s/it]


[stopped] Interrupted by user (KeyboardInterrupt).
Partial outputs saved to:
  Workflow map: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\workflow_instru_map.csv
  Run metrics:  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\run_instru_metrics.csv


In [11]:
# working dataset creation based on the run metrics and RQ2 contents

In [ ]:
import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    import yaml  # pip install pyyaml
except ImportError:
    yaml = None

try:
    from tqdm import tqdm  # pip install tqdm
except ImportError:
    tqdm = None


# ============================================================
# CONFIG
# ============================================================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

CI_SEED_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\1_CI_YML_Instru.csv"
)
GMD_SEED_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\1_Gradle_GMD_Instru.csv"
)

OUT_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_WORKFLOW_MAP_CSV = OUT_DIR / "workflow_instru_map_seeded_ci_aligned.csv"
OUT_RUN_METRICS_CSV = OUT_DIR / "run_instru_metrics_seeded_ci_aligned.csv"

DEFAULT_BRANCH_ONLY = True
FETCH_JOBS_FOR_EACH_RUN = True
FALLBACK_LIST_WORKFLOWS_FOR_GMD_HINTED_REPOS = True

MAX_RUNS_PER_WORKFLOW: Optional[int] = None
RUN_CREATED_AT_AFTER: Optional[str] = None  # ISO8601 or None

# ----------- API robustness knobs -----------
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000


# ============================================================
# CI-aligned classification patterns (fallback only)
# Naming matches your CI detector:
#   Emu_Custom, Emu_Community, Third-Party, Real Device
# Invocation types match:
#   Gradle_GMD, Gradle_Connected, Gradle, ADB, 3P CLIs
# ============================================================
EMU_COMMUNITY_ACTION_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*("
    r"reactivecircus/android-emulator-runner|"
    r"malinskiy/action-android/emulator-run-cmd|"
    r"hannesa2/action-android/emulator-run-cmd|"
    r"vgaidarji/android-github-actions-emulator"
    r")@"
)

OTHER_GHA_EMULATOR_ACTION_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*"
    r"(?!reactivecircus/android-emulator-runner@)"
    r"(?!malinskiy/action-android/emulator-run-cmd@)"
    r"(?!hannesa2/action-android/emulator-run-cmd@)"
    r"(?!emulator-wtf/run-tests@)"
    r"[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@"
)

EMU_CUSTOM_RUNTIME_RE = re.compile(
    r"(?mi)\b("
    r"emulator\b[^\n]*-avd\s+\S+|"
    r"adb\s+wait[- ]?for[- ]?device\b|"
    r"adb\s+-s\s+emulator-\d+\b|"
    r"\bandroid-wait-for-emulator\b|"
    r"\bstart-emulator\.sh\b|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b[^\n]*(system-images;android-|system-images;android)|"
    r"\bandroid\b[^\n]*\bcreate\s+avd\b"
    r")\b"
)

REAL_DEVICE_ADB_RE = re.compile(
    r"(?mi)\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b"
)

THIRD_PARTY_RE = re.compile(
    r"(?mi)\b("
    r"gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|"
    r"\bflank\s+android\s+run\b|"
    r"\bappcenter\s+test\s+run\s+android\b|"
    r"\bsaucectl\b|"
    r"\b(browserstack|bstack)\b|"
    r"\bmaestro\s+cloud\b|"
    r"\bemulator\.wtf\b|"
    r"^\s*uses\s*:\s*emulator-wtf/run-tests@"
    r")\b"
)

# Invocation detection (fallback only)
GRADLE_CONNECTED_RE = re.compile(
    r"(?mi)\b("
    r"connectedAndroidTest|connectedCheck|deviceCheck|allDevicesCheck|connectedBenchmarkAndroidTest"
    r")\b"
)
GRADLE_GMD_RE = re.compile(r"(?mi)\b(managedDeviceCheck|managedDevice|allDevicesCheck)\b")
GRADLE_BASELINEPROFILE_RE = re.compile(r"(?mi)\b(generate\w*baselineprofile|collect\w*baselineprofile)\b")
ADB_AM_INSTR_RE = re.compile(r"(?mi)\bam\s+instrument\b")


# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def b64_to_text(content_b64: str) -> Optional[str]:
    try:
        return base64.b64decode(content_b64).decode("utf-8", errors="ignore")
    except Exception:
        return None

def load_tokens_from_env_file(env_path: Path) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def dot_fullname_to_slash(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return ""
    if "/" in s:
        return s
    if "." not in s:
        return s
    owner, repo = s.split(".", 1)
    return f"{owner}/{repo}"

def extract_workflow_filename_from_seed_filename(seed_filename: str) -> Optional[str]:
    # owner.repo__github_actions++main.yml  -> main.yml
    s = (seed_filename or "").strip()
    m = re.search(r"__github_actions\+\+(.*)$", s)
    if not m:
        return None
    wf = m.group(1).strip()
    return wf or None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    if csv_path.exists():
        return
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys


# ============================================================
# GitHub API client (token rotation + retries)
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "instru-miner-seeded-ci-aligned/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException as e:
                print(f"[net] {method} {url} attempt {attempt}: {e}")
                self._backoff(attempt)
                continue

            last_status = resp.status_code
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                print(f"[error] {method} {url} -> {resp.status_code}")
                print((resp.text or "")[:400])
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# ============================================================
# Seed loading: repo->workflow files + (repo,wf)->label map
# ============================================================
def load_seed_targets(
    ci_csv: Path,
    gmd_csv: Path
) -> Tuple[Dict[str, Set[str]], Set[str], Dict[Tuple[str, str], Dict[str, str]]]:
    """
    Returns:
      repo_to_workflow_files: { 'owner/repo': {'main.yml', ...} }
      gmd_hinted_repos: set('owner/repo', ...)
      ci_labels: {(full_name, wf_file): {'Exec_Env_Style':..., 'test_invocation':..., 'instru_t_ci_signal':..., ...}}
    """
    repo_to_wf: Dict[str, Set[str]] = {}
    gmd_repos: Set[str] = set()
    ci_labels: Dict[Tuple[str, str], Dict[str, str]] = {}

    # CI seeds
    with ci_csv.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            full = dot_fullname_to_slash(row.get("full_name") or "")
            if not full:
                continue

            wf = extract_workflow_filename_from_seed_filename(row.get("filename") or "")
            if not wf:
                continue

            exec_style = (row.get("Exec_Env_Style") or "").strip()
            test_inv = (row.get("test_invocation") or "").strip()
            sig = str(row.get("instru_t_ci_signal") or "").strip().lower()
            called_sig = str(row.get("called_instru_t_ci_signal") or "").strip().lower()

            # include as a seed if it has any instrumentation evidence OR style/invocation is present
            if sig in ("true", "1", "yes") or called_sig in ("true", "1", "yes") or exec_style or test_inv:
                repo_to_wf.setdefault(full, set()).add(wf)
                ci_labels[(full, wf)] = {
                    "Exec_Env_Style": exec_style,
                    "test_invocation": test_inv,
                    "instru_t_ci_signal": sig,
                    "called_instru_t_ci_signal": called_sig,
                }

    # Gradle GMD seeds -> repo hints (repo-level)
    with gmd_csv.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            full = dot_fullname_to_slash(row.get("full_name") or "")
            if not full:
                continue
            gmd_sig = str(row.get("gmd_config_signal") or "").strip().lower()
            style = (row.get("Exec_Env_Style") or "").strip()
            if gmd_sig in ("true", "1", "yes") or style:
                gmd_repos.add(full)

    return repo_to_wf, gmd_repos, ci_labels


# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def get_workflow_meta(gh: GitHubClient, full_name: str, workflow_file: str) -> Optional[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_file}"
    return gh.request_json("GET", url)

def fetch_workflow_yaml_text(gh: GitHubClient, full_name: str, workflow_path: str, ref: Optional[str]) -> Optional[str]:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    params = {"ref": ref} if ref else None
    data = gh.request_json("GET", url, params=params)
    if not data or not isinstance(data, dict):
        return None

    if data.get("encoding") == "base64" and data.get("content"):
        return b64_to_text(data["content"])

    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text
        except requests.exceptions.RequestException:
            return None
    return None

def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    return list(gh.paginate(url, params={}, item_key="workflows"))

def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_identifier: str, branch: Optional[str]) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_identifier}/runs"
    params = {"branch": branch} if branch else {}
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))

def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


# ============================================================
# Fallback classification (only used if CI seed didn’t label this workflow)
# ============================================================
def classify_workflow_ci_aligned(yaml_text: str) -> Tuple[Set[str], Set[str], bool]:
    styles: Set[str] = set()
    inv: Set[str] = set()

    # env style
    if EMU_CUSTOM_RUNTIME_RE.search(yaml_text):
        styles.add("Emu_Custom")

    if EMU_COMMUNITY_ACTION_RE.search(yaml_text) or OTHER_GHA_EMULATOR_ACTION_RE.search(yaml_text):
        styles.add("Emu_Community")

    if THIRD_PARTY_RE.search(yaml_text):
        styles.add("Third-Party")

    if REAL_DEVICE_ADB_RE.search(yaml_text):
        styles.add("Real Device")

    # invocation
    if GRADLE_GMD_RE.search(yaml_text):
        inv.add("Gradle_GMD")
    if GRADLE_CONNECTED_RE.search(yaml_text):
        inv.add("Gradle_Connected")
    if GRADLE_BASELINEPROFILE_RE.search(yaml_text):
        inv.add("Gradle")
    if ADB_AM_INSTR_RE.search(yaml_text):
        inv.add("ADB")
    if THIRD_PARTY_RE.search(yaml_text):
        inv.add("3P CLIs")

    looks_like_instru = bool(styles or inv)
    if not inv:
        inv.add("UNKNOWN")

    return styles, inv, looks_like_instru


# ============================================================
# Jobs inference (unchanged)
# ============================================================
INSTRU_STEP_NAME_RE = re.compile(
    r"(instrument|connected.*androidtest|androidtest|manageddevice|gmd|emulator runner|"
    r"firebase test|test lab|device farm|uiautomator|espresso)",
    re.IGNORECASE,
)
INSTRU_JOB_NAME_RE = re.compile(
    r"(instrument|androidtest|connected|manageddevice|gmd|emulator|firebase|test lab|device farm)",
    re.IGNORECASE,
)

def infer_instru_from_jobs(jobs: List[Dict]) -> Tuple[str, str, Optional[int], Optional[int], str]:
    if not jobs:
        return "unknown", "none", None, None, ""

    starts, ends = [], []
    labels: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt: starts.append(sdt)
        if edt: ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels.add(lab.strip())

    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)

    for j in jobs:
        job_name = j.get("name") or ""
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []
        for st in steps:
            step_name = (st.get("name") or "")
            if INSTRU_STEP_NAME_RE.search(step_name) or INSTRU_JOB_NAME_RE.search(job_name):
                concl = (st.get("conclusion") or st.get("status") or "unknown")
                sd = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
                if sd is None:
                    sd = dt_to_seconds(iso_to_dt(j.get("started_at")), iso_to_dt(j.get("completed_at")))
                return (concl, "step", sd, run_dur, ",".join(sorted(labels)))

    for j in jobs:
        job_name = j.get("name") or ""
        if INSTRU_JOB_NAME_RE.search(job_name):
            concl = (j.get("conclusion") or "unknown")
            jd = dt_to_seconds(iso_to_dt(j.get("started_at")), iso_to_dt(j.get("completed_at")))
            return (concl, "job", jd, run_dur, ",".join(sorted(labels)))

    return ("unknown", "none", None, run_dur, ",".join(sorted(labels)))


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens)

    repo_to_wf_files, gmd_hinted_repos, ci_labels = load_seed_targets(CI_SEED_CSV, GMD_SEED_CSV)
    seed_repos = sorted(set(repo_to_wf_files.keys()) | set(gmd_hinted_repos))
    if not seed_repos:
        raise RuntimeError("No repos found from seed files. Check CSV paths/columns.")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    workflow_fields = [
        "full_name",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "looks_like_instru",
        "styles",
        "invocation_types",
        "label_source",  # CI_SEED or YAML_FALLBACK
        "scanned_at_utc",
    ]
    run_fields = [
        "full_name",
        "default_branch",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "run_id",
        "run_number",
        "head_sha",
        "created_at",
        "run_started_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "instru_conclusion",
        "instru_detect_method",
        "instru_duration_seconds",
        "run_duration_seconds",
        "runner_labels_union",
        "extracted_at_utc",
    ]

    ensure_csv_header(OUT_WORKFLOW_MAP_CSV, workflow_fields)
    ensure_csv_header(OUT_RUN_METRICS_CSV, run_fields)
    existing_run_ids = load_existing_keys(OUT_RUN_METRICS_CSV, "run_id")

    repo_iter = seed_repos
    if tqdm is not None:
        repo_iter = tqdm(seed_repos, desc="Repos (seeded CI-aligned)")

    default_branch_cache: Dict[str, str] = {}

    for full_name in repo_iter:
        # default branch
        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        wf_identifiers = set(repo_to_wf_files.get(full_name, set()))

        # fallback list workflows if repo is GMD-hinted but no CI workflows were found
        if not wf_identifiers and (full_name in gmd_hinted_repos) and FALLBACK_LIST_WORKFLOWS_FOR_GMD_HINTED_REPOS:
            for wf in list_workflows(gh, full_name) or []:
                wf_identifiers.add((wf.get("path") or "").split("/")[-1])

        for wf_file in sorted(wf_identifiers):
            meta = get_workflow_meta(gh, full_name, wf_file)
            if not meta or not isinstance(meta, dict):
                continue

            wf_id = str(meta.get("id") or "").strip()
            wf_name = (meta.get("name") or "").strip()
            wf_path = (meta.get("path") or "").strip()
            if not wf_path:
                continue

            ytext = fetch_workflow_yaml_text(
                gh, full_name, wf_path, ref=default_branch if DEFAULT_BRANCH_ONLY else None
            ) or ""

            # ===== label source priority: CI seed > fallback regex =====
            seed = ci_labels.get((full_name, wf_file))
            if seed:
                styles_str = (seed.get("Exec_Env_Style") or "").strip()
                inv_str = (seed.get("test_invocation") or "").strip()
                sig = (seed.get("instru_t_ci_signal") or "").strip().lower()
                called_sig = (seed.get("called_instru_t_ci_signal") or "").strip().lower()

                styles = {s.strip() for s in styles_str.split(",") if s.strip()}
                inv = {s.strip() for s in inv_str.split(",") if s.strip()}

                looks_like = bool(styles or inv or sig in ("true", "1", "yes") or called_sig in ("true", "1", "yes"))
                if not inv:
                    inv = {"UNKNOWN"}
                label_source = "CI_SEED"
            else:
                styles, inv, looks_like = classify_workflow_ci_aligned(ytext)
                label_source = "YAML_FALLBACK"

            append_row(OUT_WORKFLOW_MAP_CSV, workflow_fields, {
                "full_name": full_name,
                "workflow_identifier": wf_file,
                "workflow_id": wf_id,
                "workflow_name": wf_name,
                "workflow_path": wf_path,
                "looks_like_instru": "yes" if looks_like else "no",
                "styles": ",".join(sorted(styles)),
                "invocation_types": ",".join(sorted(inv)),
                "label_source": label_source,
                "scanned_at_utc": now_utc_iso(),
            })

            # runs
            runs = list_workflow_runs(
                gh, full_name, wf_file, branch=default_branch if DEFAULT_BRANCH_ONLY else None
            )
            if MAX_RUNS_PER_WORKFLOW is not None:
                runs = runs[:MAX_RUNS_PER_WORKFLOW]

            for run in runs:
                run_id = str(run.get("id") or "").strip()
                if not run_id or run_id in existing_run_ids:
                    continue

                created_at = run.get("created_at") or ""
                if after_dt:
                    cdt = iso_to_dt(created_at)
                    if cdt and cdt < after_dt:
                        continue

                head_branch = run.get("head_branch") or ""
                head_sha = run.get("head_sha") or ""
                if DEFAULT_BRANCH_ONLY and head_branch != default_branch:
                    continue

                instru_conc = "unknown"
                detect_method = "none"
                instru_dur = None
                run_dur = None
                runner_labels = ""

                if FETCH_JOBS_FOR_EACH_RUN:
                    jobs = list_run_jobs(gh, full_name, int(run_id))
                    instru_conc, detect_method, instru_dur, run_dur, runner_labels = infer_instru_from_jobs(jobs)

                append_row(OUT_RUN_METRICS_CSV, run_fields, {
                    "full_name": full_name,
                    "default_branch": default_branch,
                    "workflow_identifier": wf_file,
                    "workflow_id": wf_id,
                    "workflow_name": wf_name,
                    "workflow_path": wf_path,
                    "run_id": run_id,
                    "run_number": run.get("run_number") or "",
                    "head_sha": head_sha,
                    "created_at": created_at,
                    "run_started_at": run.get("run_started_at") or "",
                    "status": run.get("status") or "",
                    "run_conclusion": run.get("conclusion") or "",
                    "event": run.get("event") or "",
                    "head_branch": head_branch,
                    "html_url": run.get("html_url") or "",
                    "instru_conclusion": instru_conc,
                    "instru_detect_method": detect_method,
                    "instru_duration_seconds": "" if instru_dur is None else str(instru_dur),
                    "run_duration_seconds": "" if run_dur is None else str(run_dur),
                    "runner_labels_union": runner_labels,
                    "extracted_at_utc": now_utc_iso(),
                })
                existing_run_ids.add(run_id)

    print("Done.")
    print("Workflow map:", OUT_WORKFLOW_MAP_CSV)
    print("Run metrics :", OUT_RUN_METRICS_CSV)


if __name__ == "__main__":
    main()


[load] episodes: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Change_Episodes.csv
[load] boundary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
[load] runs    : C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\run_instru_metrics.csv


C:\Users\gilla\AppData\Local\Temp\ipykernel_58564\3365052840.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_Community' 'Emu_Community' 'Emu_Community'
 'Emu_Community' 'Emu_

[save] C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\RQ4_Datasets\rq4_runs_labeled.csv
[save] C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\RQ4_Datasets\rq4_episode_performance.csv
[save] C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\RQ4_Datasets\rq4_style_overall_performance.csv


c:\GitHub\.PCvenv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\gilla\AppData\Local\Temp\ipykernel_58564\3365052840.py:275: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: aggregate_metrics(g))
C:\Users\gilla\AppData\Local\Temp\ipykernel_58564\3365052840.py:288: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence 

ValueError: columns overlap but no suffix specified: Index(['n_runs', 'n_concluded', 'success_rate', 'failure_rate', 'cancel_rate',
       'median_run_duration_s', 'p90_run_duration_s', 'median_queue_s',
       'p90_queue_s', 'n_instru_detected', 'instru_success_rate',
       'instru_failure_rate', 'median_instru_duration_s'],
      dtype='object')